In [1]:
import pandas as pd
from scipy.io import mmread
import scipy.sparse as sp
import torch
import torch.nn as nn
from torch.optim import Adam
import scipy.sparse as sp
import numpy as np
from sentence_transformers import SentenceTransformer
import ast
import os
import matplotlib.pyplot as plt

c:\Users\alex\Desktop\cours polymtl\log6308\tp4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ids = pd.read_csv('data/TP4-ids.csv', index_col=0)
articles = pd.concat([
    pd.read_csv('data/TP4-articles1.csv', index_col=0),
    pd.read_csv('data/TP4-articles2.csv', index_col=0)
], ignore_index=True)

adjacency = mmread('data/TP4-matrice-adjacence.dgt').tocsr() # Shape : (67294, 67294)

In [20]:
def generate_recommendations(model, test_ids, ids, adjacency, device='cuda', top_k=20):
    '''
    test_ids : DataFrame de test
    ids      : DataFrame de correspondance n <-> id
    '''
    model.eval()
    model = model.to(device)
    model.L = model.L.to(device)
    
    idx_to_hexid = ids.set_index('n')['id'].to_dict()
    
    with torch.no_grad():
        E_concat = model()  # (n_items, embed_size * (n_layers+1))
    
    results = []
    
    for _, row in test_ids.iterrows():
        hex_id = row['id']
        idx = row['idx']
        i = idx - 1 
        
        scores = (E_concat @ E_concat[i])  # (n_items,)
        

        # on retire les articles déjà cités par i de la liste de recommandations et l'article lui-même
        known_cited = adjacency.getrow(i).indices
        scores[known_cited] = -float('inf')
        scores[i] = -float('inf')
        
        # Top 20
        top20_indices = scores.argsort(descending=True)[:top_k].cpu().numpy()
        
        top20_hexids = [idx_to_hexid[idx + 1] for idx in top20_indices]  # +1 car ids est 1-based
        
        results.append({
            'id': hex_id,
            'recommandations': ' '.join(top20_hexids)
        })
    
    return pd.DataFrame(results)

# NGCF

In [11]:
class NGCF_ItemItem(nn.Module):
    def __init__(self, n_items, adjacency, n_layers=3, embed_size=256, dropout=0):
        super().__init__()
        self.n_items = n_items
        self.n_layers = n_layers
        self.embed_size = embed_size
        self.dropout = dropout


        # l'embedding initial est juste un paramètre du modèle appris grâce au contenu collaboratif
        self.E0 = nn.Parameter(torch.empty(n_items, embed_size))
        nn.init.xavier_uniform_(self.E0)

        
        # Matrices W pour chaque couches de propagations
        self.W1 = nn.ModuleList([nn.Linear(embed_size, embed_size, bias=False) for _ in range(n_layers)])
        self.W2 = nn.ModuleList([nn.Linear(embed_size, embed_size, bias=False) for _ in range(n_layers)])
        
        # Si on normalise pas entre les couches les valeurs explosent et ça diverge donc on ajoute une normalisation après chaque couche
        self.norms = nn.ModuleList([nn.LayerNorm(embed_size) for _ in range(n_layers)])

        self.L = self.normalize(adjacency)  # (L = D^-1/2 · A · D^-1/2)

    def normalize(self, A):
        degrees = np.array(A.sum(axis=1)).flatten()
        D_inv_sqrt = sp.diags(1.0 / np.sqrt(degrees + 1e-8))
        L = D_inv_sqrt @ A @ D_inv_sqrt

        # on garde la matrice creuse pour les perfs
        L_coo = L.tocoo()
        indices = torch.tensor([L_coo.row, L_coo.col], dtype=torch.long)
        values = torch.tensor(L_coo.data, dtype=torch.float32)
        return torch.sparse_coo_tensor(indices, values, L.shape).coalesce()

    def forward(self):


        # on utilise l'autocast pour réduire la précision à float16 et économiser de la VRAM, car les matrices sont grandes et les calculs lourds. Cela peut accélérer l'entraînement sur GPU tout en gardant une précision suffisante pour ce type de tâche.
        # # sparse.mm ne supporte pas float16 sur CUDA donc on fait la propagation en float32 et on cast après
        E = self.E0
        all_embeddings = [E]
        for l in range(self.n_layers):
            LE = torch.sparse.mm(self.L, E.float())
            LE = nn.functional.normalize(LE, p=2, dim=1)
            with torch.autocast(device_type='cuda', dtype=torch.float16): 
                # on propage les embeddings à travers le graphe
                # (L+I)E.W1  +  L(E°E).W2
                term1 = self.W1[l](LE + E)           # (L+I)E.W1
                term2 = self.W2[l](LE * E)            # L(E°E).W2
                E = nn.functional.leaky_relu(term1 + term2)
                E = self.norms[l](E)
                E = nn.functional.dropout(E, p=self.dropout, training=self.training)
            all_embeddings.append(E)
            
        # Concatenation de toutes les couches
        E_concat = torch.cat(all_embeddings, dim=1)
        return E_concat

    def bpr_loss(self, E_concat, i, j, k):
        # i cite j (positif), i ne cite pas k (négatif)
        e_i = E_concat[i]
        e_j = E_concat[j]
        e_k = E_concat[k]
        
        pos_score = (e_i * e_j).sum(dim=1)
        neg_score = (e_i * e_k).sum(dim=1)
        
        loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()
        return loss
    

def get_training_pairs(adjacency):
    '''
    Extrait toutes les paires (i, j) positives depuis la matrice d'adjacence
    '''
    adjacency_coo = adjacency.tocoo()
    rows = adjacency_coo.row  # articles citants
    cols = adjacency_coo.col  # articles cités
    return rows, cols


def train(model, adjacency, n_epochs=50, batch_size=1024, lr=1e-4, lambda_reg=1e-5, checkpoint_path='checkpoint.pt'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.L = model.L.to(device)

    torch.cuda.empty_cache()
    scaler = torch.amp.GradScaler() # pour gérer l'autocast et éviter les problèmes de sous-flux de gradients en float16    
    optimizer = Adam(model.parameters(), lr=lr)
    

    start_epoch = 0
    loss_history = []
    if os.path.exists(checkpoint_path):
        print(f"Checkpoint trouvé, reprise depuis {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scaler.load_state_dict(checkpoint['scaler_state'])
        start_epoch = checkpoint['epoch'] + 1
        loss_history = checkpoint['loss_history']
        print(f"Reprise à l'epoch {start_epoch + 1}/{n_epochs}")



    # Extraire toutes les paires positives
    rows, cols = get_training_pairs(adjacency)
    n_pairs = len(rows)
    n_items = adjacency.shape[0]
    
    print(f"Nb paires positives : {n_pairs}")
    print(f"Training sur : {device}")

    for epoch in range(start_epoch, n_epochs):
        print(f"Epoch {epoch+1}/{n_epochs}")
        model.train()

        # on shuffle les paires à chaque époque
        perm = np.random.permutation(n_pairs)
        rows_shuffled = rows[perm]
        cols_shuffled = cols[perm]
        
        total_loss = 0
        n_batches = 0
        for start in range(0, n_pairs, batch_size):
            print(f"\rBatch {start//batch_size + 1}/{(n_pairs + batch_size - 1) // batch_size}",end="", flush=True)
            i_batch = rows_shuffled[start:start + batch_size]
            j_batch = cols_shuffled[start:start + batch_size]
            k_batch = np.random.randint(0, n_items, size=len(i_batch)) # quasiment sur que k_batch ne contient pas de paires positives (i, k) car la matrice est très creuse, mais même si c'était le cas ça ne changerait rien à l'entrainement par loi des grands nombres.
            
            i_t = torch.tensor(i_batch, dtype=torch.long).to(device)
            j_t = torch.tensor(j_batch, dtype=torch.long).to(device)
            k_t = torch.tensor(k_batch, dtype=torch.long).to(device)
            
            optimizer.zero_grad()
            E_concat = model()

            # print(f"E_concat NaN: {torch.isnan(E_concat).sum()}, Inf: {torch.isinf(E_concat).sum()}")
            # print(f"E_concat min/max: {E_concat.min():.4f} / {E_concat.max():.4f}")

            
            # BPR loss avec régularisation L2
            loss_bpr = model.bpr_loss(E_concat, i_t, j_t, k_t) 
            reg_embeddings = (model.E0[i_t].norm(2).pow(2) + model.E0[j_t].norm(2).pow(2) + model.E0[k_t].norm(2).pow(2)) / len(i_t)
            reg_weights = 0
            for name, param in model.named_parameters():
                if 'W' in name:
                    reg_weights += param.norm(2).pow(2)
            loss = loss_bpr + lambda_reg * (reg_embeddings + reg_weights)

            # backward et update
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            del E_concat
            torch.cuda.empty_cache()
            
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1}/{n_epochs} — Loss: {avg_loss:.4f}")
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scaler_state': scaler.state_dict(),
            'loss': avg_loss,
            'loss_history': loss_history
        }, checkpoint_path)
        print(f"Checkpoint sauvegardé (epoch {epoch+1})")

    
    return model, loss_history

In [18]:
model = NGCF_ItemItem(
        n_items=adjacency.shape[0],
        adjacency=adjacency, 
        n_layers=3, 
        embed_size=128, 
        dropout=0.2
    )
trained_model, loss_history = train(model, adjacency, n_epochs=30, batch_size=2048, lr=1e-4, lambda_reg=1e-5, checkpoint_path='checkpoint_NGCF_noInit.pt')

Checkpoint trouvé, reprise depuis checkpoint_NGCF_noInit.pt
Reprise à l'epoch 12/30
Nb paires positives : 621195
Training sur : cuda
Epoch 12/30
Batch 304/304Epoch 12/30 — Loss: 0.6871
Checkpoint sauvegardé (epoch 12)
Epoch 13/30
Batch 304/304Epoch 13/30 — Loss: 0.6517
Checkpoint sauvegardé (epoch 13)
Epoch 14/30
Batch 304/304Epoch 14/30 — Loss: 0.6277
Checkpoint sauvegardé (epoch 14)
Epoch 15/30
Batch 304/304Epoch 15/30 — Loss: 0.6004
Checkpoint sauvegardé (epoch 15)
Epoch 16/30
Batch 304/304Epoch 16/30 — Loss: 0.5794
Checkpoint sauvegardé (epoch 16)
Epoch 17/30
Batch 304/304Epoch 17/30 — Loss: 0.5562
Checkpoint sauvegardé (epoch 17)
Epoch 18/30
Batch 304/304Epoch 18/30 — Loss: 0.5444
Checkpoint sauvegardé (epoch 18)
Epoch 19/30
Batch 304/304Epoch 19/30 — Loss: 0.5125
Checkpoint sauvegardé (epoch 19)
Epoch 20/30
Batch 304/304Epoch 20/30 — Loss: 0.5066
Checkpoint sauvegardé (epoch 20)
Epoch 21/30
Batch 304/304Epoch 21/30 — Loss: 0.4892
Checkpoint sauvegardé (epoch 21)
Epoch 22/30
Batch

In [21]:
model = NGCF_ItemItem(n_items=adjacency.shape[0], adjacency=adjacency, n_layers=3, embed_size=128, dropout=0.2)
checkpoint_path = 'checkpoint_NGCF_noInit.pt'

if os.path.exists(checkpoint_path):
    print(f"Checkpoint trouvé {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location='cuda')
    model.load_state_dict(checkpoint['model_state'])


test_ids = pd.read_csv('data/TP4-ids-test.csv', index_col=0)
recommendations = generate_recommendations(model, test_ids, ids, adjacency)
recommendations.to_csv('soumission_NGCF_noInit_30epoch.csv', index=False)
print(recommendations.head())


Checkpoint trouvé checkpoint_NGCF_noInit.pt
                         id                                    recommandations
0  56d87c8bdabfae2eee45fce6  558b3daa84ae84d265c246c1 53e998bfb7602d97020f6...
1  53e9b7d9b7602d9704389448  53e9b857b7602d970441c4d8 53e9b89ab7602d9704471...
2  53e99e04b7602d97026ca90e  53e9b708b7602d970429edd1 53e9b4efb7602d970401c...
3  53e9b32bb7602d9703dfdacf  53e9b145b7602d9703bcbadc 53e99e6ab7602d970272e...
4  53e9b9adb7602d970459ef8c  53e9b791b7602d970433a23e 53e9a423b7602d9702d3c...


# LightGCN

In [15]:
class LightGCN_ItemItem(nn.Module):
    def __init__(self, n_items, adjacency, n_layers=3, embed_size=256, dropout=0):
        super().__init__()
        self.n_items = n_items
        self.n_layers = n_layers
        self.embed_size = embed_size
        self.dropout = dropout


        # l'embedding initial est juste un paramètre du modèle appris grâce au contenu collaboratif
        self.E0 = nn.Parameter(torch.empty(n_items, embed_size))
        nn.init.xavier_uniform_(self.E0)

        self.L = self.normalize(adjacency) 

    def normalize(self, A):
        degrees = np.array(A.sum(axis=1)).flatten()
        D_inv_sqrt = sp.diags(1.0 / np.sqrt(degrees + 1e-8))
        L = D_inv_sqrt @ A @ D_inv_sqrt

        L_coo = L.tocoo()
        indices = torch.tensor([L_coo.row, L_coo.col], dtype=torch.long)
        values = torch.tensor(L_coo.data, dtype=torch.float32)
        return torch.sparse_coo_tensor(indices, values, L.shape).coalesce()


    def forward(self):

        E = self.E0
        all_embeddings = [E]
        for l in range(self.n_layers):
            # Juste L·E — pas de W, pas de non-linéarité
            E = torch.sparse.mm(self.L, E)
            all_embeddings.append(E)

        # Moyenne pondérée au lieu de concaténation
        E_final = torch.stack(all_embeddings, dim=0).mean(dim=0)
        return E_final

    def bpr_loss(self, E_final, i, j, k):
        e_i = E_final[i]
        e_j = E_final[j]
        e_k = E_final[k]

        pos_score = (e_i * e_j).sum(dim=1)
        neg_score = (e_i * e_k).sum(dim=1)

        return -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()



def train_lightGCN(model, adjacency, n_epochs=50, batch_size=1024, lr=1e-4, lambda_reg=1e-5, checkpoint_path='checkpoint.pt'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.L = model.L.to(device)

    torch.cuda.empty_cache()
    scaler = torch.amp.GradScaler() # pour gérer l'autocast et éviter les problèmes de sous-flux de gradients en float16    
    optimizer = Adam(model.parameters(), lr=lr)
    

    start_epoch = 0
    loss_history = []
    if os.path.exists(checkpoint_path):
        print(f"Checkpoint trouvé, reprise depuis {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scaler.load_state_dict(checkpoint['scaler_state'])
        start_epoch = checkpoint['epoch'] + 1
        loss_history = checkpoint['loss_history']
        print(f"Reprise à l'epoch {start_epoch + 1}/{n_epochs}")



    # Extraire toutes les paires positives
    rows, cols = get_training_pairs(adjacency)
    n_pairs = len(rows)
    n_items = adjacency.shape[0]
    
    print(f"Nb paires positives : {n_pairs}")
    print(f"Training sur : {device}")

    for epoch in range(start_epoch, n_epochs):
        print(f"Epoch {epoch+1}/{n_epochs}")
        model.train()

        # on shuffle les paires à chaque époque
        perm = np.random.permutation(n_pairs)
        rows_shuffled = rows[perm]
        cols_shuffled = cols[perm]
        
        total_loss = 0
        n_batches = 0
        for start in range(0, n_pairs, batch_size):
            print(f"\rBatch {start//batch_size + 1}/{(n_pairs + batch_size - 1) // batch_size}",end="", flush=True)
            i_batch = rows_shuffled[start:start + batch_size]
            j_batch = cols_shuffled[start:start + batch_size]
            k_batch = np.random.randint(0, n_items, size=len(i_batch)) # quasiment sur que k_batch ne contient pas de paires positives (i, k) car la matrice est très creuse, mais même si c'était le cas ça ne changerait rien à l'entrainement par loi des grands nombres.
            
            i_t = torch.tensor(i_batch, dtype=torch.long).to(device)
            j_t = torch.tensor(j_batch, dtype=torch.long).to(device)
            k_t = torch.tensor(k_batch, dtype=torch.long).to(device)
            
            optimizer.zero_grad()
            E_final = model()

            # print(f"E_concat NaN: {torch.isnan(E_concat).sum()}, Inf: {torch.isinf(E_concat).sum()}")
            # print(f"E_concat min/max: {E_concat.min():.4f} / {E_concat.max():.4f}")

            
            # BPR loss avec régularisation L2
            loss_bpr = model.bpr_loss(E_final, i_t, j_t, k_t) 
            reg_embeddings = (model.E0[i_t].norm(2).pow(2) + model.E0[j_t].norm(2).pow(2) + model.E0[k_t].norm(2).pow(2)) / len(i_t)
            loss = loss_bpr + lambda_reg * reg_embeddings

            # backward et update
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            del E_final
            torch.cuda.empty_cache()
            
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1}/{n_epochs} — Loss: {avg_loss:.4f}")
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scaler_state': scaler.state_dict(),
            'loss': avg_loss,
            'loss_history': loss_history
        }, checkpoint_path)
        print(f"Checkpoint sauvegardé (epoch {epoch+1})")

    
    return model, loss_history

In [19]:
model = LightGCN_ItemItem(
        n_items=adjacency.shape[0],
        adjacency=adjacency, 
        n_layers=3, 
        embed_size=128, 
        dropout=0.2
    )
trained_model, loss_history = train_lightGCN(model, adjacency, n_epochs=30, batch_size=2048, lr=1e-4, lambda_reg=1e-5, checkpoint_path='checkpoint_lightGCN_noInit.pt')

Checkpoint trouvé, reprise depuis checkpoint_lightGCN_noInit.pt
Reprise à l'epoch 3/30
Nb paires positives : 621195
Training sur : cuda
Epoch 3/30
Batch 304/304Epoch 3/30 — Loss: 1.4518
Checkpoint sauvegardé (epoch 3)
Epoch 4/30
Batch 304/304Epoch 4/30 — Loss: 1.4161
Checkpoint sauvegardé (epoch 4)
Epoch 5/30
Batch 304/304Epoch 5/30 — Loss: 1.3762
Checkpoint sauvegardé (epoch 5)
Epoch 6/30
Batch 304/304Epoch 6/30 — Loss: 1.3452
Checkpoint sauvegardé (epoch 6)
Epoch 7/30
Batch 304/304Epoch 7/30 — Loss: 1.3355
Checkpoint sauvegardé (epoch 7)
Epoch 8/30
Batch 304/304Epoch 8/30 — Loss: 1.3407
Checkpoint sauvegardé (epoch 8)
Epoch 9/30
Batch 304/304Epoch 9/30 — Loss: 1.3106
Checkpoint sauvegardé (epoch 9)
Epoch 10/30
Batch 304/304Epoch 10/30 — Loss: 1.2971
Checkpoint sauvegardé (epoch 10)
Epoch 11/30
Batch 304/304Epoch 11/30 — Loss: 1.2968
Checkpoint sauvegardé (epoch 11)
Epoch 12/30
Batch 304/304Epoch 12/30 — Loss: 1.2904
Checkpoint sauvegardé (epoch 12)
Epoch 13/30
Batch 304/304Epoch 13/3

In [22]:
model = LightGCN_ItemItem(n_items=adjacency.shape[0],adjacency=adjacency, n_layers=3, embed_size=128, dropout=0.2)
checkpoint_path = 'checkpoint_lightGCN_noInit.pt'

if os.path.exists(checkpoint_path):
    print(f"Checkpoint trouvé {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location='cuda')
    model.load_state_dict(checkpoint['model_state'])


test_ids = pd.read_csv('data/TP4-ids-test.csv', index_col=0)
recommendations = generate_recommendations(model, test_ids, ids, adjacency)
recommendations.to_csv('soumission_lightGCN_noInit_30epoch.csv', index=False)
print(recommendations.head())


Checkpoint trouvé checkpoint_lightGCN_noInit.pt
                         id                                    recommandations
0  56d87c8bdabfae2eee45fce6  5550463245ce0a409eb5caf0 53e9b961b7602d970454c...
1  53e9b7d9b7602d9704389448  558a9da7e4b0b32fcb37d87f 53e9bcb3b7602d9704936...
2  53e99e04b7602d97026ca90e  53e9a855b7602d970319d0c2 53e9a2cfb7602d9702bdb...
3  53e9b32bb7602d9703dfdacf  53e9b54ab7602d9704081d4d 53e99fa9b7602d970287e...
4  53e9b9adb7602d970459ef8c  53e9a289b7602d9702b8f511 53e9acd3b7602d97036b2...


# Dist cosinus entre Embedding specter

In [24]:
# On commence par créer les embedding qui vont encoder le contenu des articles (titre, auteurs, année, abstract). On peut utiliser un modèle pré-entraîné de type Sentence-BERT pour cela. On utilise allenai-specter (https://arxiv.org/pdf/2004.07180) qui est entraîné spécifiquement sur des articles scientifiques. On concatène le titre et l'abstract pour créer un texte représentatif de chaque article, puis on encode ce texte pour obtenir une matrice d'embeddings initiale E0 de dimension (n_articles, 768).


def parse_authors(authors_str):
    '''Extrait les noms d'auteurs depuis la string de la colonne authors'''
    if pd.isna(authors_str) or authors_str == '':
        return ''
    try:
        authors_list = ast.literal_eval(authors_str)
        names = [a.get('name', '') for a in authors_list if isinstance(a, dict)]
        return ', '.join(names)
    except:
        return ''
    
def build_text(row):
    title    = '' if pd.isna(row.get('title'))    else str(row['title'])
    year     = '' if pd.isna(row.get('year'))     else str(int(row['year']))
    abstract = '' if pd.isna(row.get('abstract')) else str(row['abstract'])
    authors  = parse_authors(row.get('authors'))
    
    # Format SPECTER : "titre [SEP] contexte" et on met les auteurs et l'année dans la partie "contexte"
    context = f"{authors} {year} {abstract}".strip()
    return f"{title} [SEP] {context}"



def build_E0_aligned(articles, ids, model_name='allenai-specter'):
    '''
    Construit E0 dans le bon ordre : E0[i] correspond à l'article n=i+1
    '''

    model = SentenceTransformer(model_name)
    n_items = ids['n'].max()
    
    # Index articles par hex id pour lookup rapide
    articles_indexed = articles.set_index('id')
    
    texts = []
    missing = 0
    
    for _, row in ids.sort_values('n').iterrows():
        hex_id = row['id']
        
        if hex_id in articles_indexed.index:
            article = articles_indexed.loc[hex_id]
            text = build_text(article)
        else:
            text = ''
            missing += 1
        
        texts.append(text)
    
    print(f"proportion d'articles non présent dans les fichier articles.csv : {missing}/{n_items}")
    
    E0 = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    # Mettre à zéro les embeddings des articles sans texte
    empty_mask = [t == '' for t in texts]
    E0[empty_mask] = 0.0
    
    return E0  # shape (67294, embed_size)

In [25]:
if os.path.exists('E0_specter.npy'):
    E0 = np.load('E0_specter.npy')
    print(f"E0 chargé depuis le cache : {E0.shape}")
else:
    E0 = build_E0_aligned(articles, ids)
    np.save('E0_specter.npy', E0)
    print(f"E0 calculé et sauvegardé : {E0.shape}")

np.save('E0_specter.npy', E0)

E0 chargé depuis le cache : (67294, 768)


In [27]:
def generate_recommendations_embedding(E0, test_ids, ids, adjacency, top_k=20):
    '''
    test_ids : DataFrame de test
    ids      : DataFrame de correspondance n <-> id
    '''
    idx_to_hexid = ids.set_index('n')['id'].to_dict()
    
    
    results = []
    
    for _, row in test_ids.iterrows():
        hex_id = row['id']
        idx = row['idx']
        i = idx - 1 
        
        scores = (E0 @ E0[i])  # (n_items,)
        

        # on retire les articles déjà cités par i de la liste de recommandations et l'article lui-même
        known_cited = adjacency.getrow(i).indices
        scores[known_cited] = -float('inf')
        scores[i] = -float('inf')
        
        # Top 20
        top20_indices = np.argsort(scores)[::-1][:top_k]
        
        top20_hexids = [idx_to_hexid[idx + 1] for idx in top20_indices]  # +1 car ids est 1-based
        
        results.append({
            'id': hex_id,
            'recommandations': ' '.join(top20_hexids)
        })
    
    return pd.DataFrame(results)


test_ids = pd.read_csv('data/TP4-ids-test.csv', index_col=0)
recommendations = generate_recommendations_embedding(E0, test_ids, ids, adjacency)
recommendations.to_csv('soumission_specter_noTraining.csv', index=False)
print(recommendations.head())

                         id                                    recommandations
0  56d87c8bdabfae2eee45fce6  53e9bcd9b7602d97049602cb 53e9bc68b7602d97048e0...
1  53e9b7d9b7602d9704389448  53e9b56cb7602d97040a44ec 5550413145ce0a409eb39...
2  53e99e04b7602d97026ca90e  53e9a4c0b7602d9702de31c9 53e9b6bfb7602d9704246...
3  53e9b32bb7602d9703dfdacf  53e9a2d5b7602d9702bde86b 53e9a97ab7602d97032d1...
4  53e9b9adb7602d970459ef8c  53e9be57b7602d9704b1a127 53e9aad8b7602d9703457...
